## FLOOD DETECTION SCENARIO PIPELINE

In [9]:
import digitalhub as dh
PROJECT_NAME = "flood-detection"
proj = dh.get_or_create_project(PROJECT_NAME) # source="git://github.com/scc-digitalhu

### Download Sentinel 2

In [3]:
# THIS NEED TO BE EXECUTED JUST ONCE
secret0 = proj.new_secret(name="CDSETOOL_ESA_USER", secret_value="esa_username")
secret1 = proj.new_secret(name="CDSETOOL_ESA_PASSWORD", secret_value="esa_password")

In [ ]:
function_s2 = proj.new_function("download_images_s2",kind="container",image="ghcr.io/tn-aixpa/sentinel-tools:0.11.5",command="python")

### Download Sentinel 1

In [ ]:
function_s1 = proj.new_function("download_images_s1",kind="container",image="ghcr.io/tn-aixpa/sentinel-tools:0.11.5",command="python")

### Log artifact

In [ ]:
proj.log_artifact(name='Lakes_TN', kind='artifact', source='Lakes_TN')
proj.log_artifact(name='Rivers_TN', kind='artifact', source='Rivers_TN')
proj.log_artifact(name='Slopes_TN', kind='artifact', source='Slopes_TN')

### Elaboration

In [ ]:
function_rs = proj.new_function("elaborate",kind="container", image="ghcr.io/tn-aixpa/rs-flood-mapping:2.6.0_b9", code_src="launch.sh")

### Pipeline

In [ ]:
%%writefile "flood_pipeline.py"

from digitalhub_runtime_kfp.dsl import pipeline_context
import datetime

def myhandler(geometry, outputName, floodDate, s1_preFloodDate, s1_postFloodDate, s2_preFloodDate, s2_postFloodDate):
  
    string_dict_data_s1Pre =  """{"satelliteParams": {"satelliteType": "Sentinel1","processingLevel": "LEVEL1","sensorMode": "IW","productType": "GRD"},"startDate":\"""" + str(s1_preFloodDate) + """\","endDate": \"""" + str(floodDate) + """\","geometry": \"""" + str(geometry) + """\","area_sampling": "True","tmp_path_same_folder_dwl":"True","artifact_name": "sentinel1_GRD_preflood"}"""
    string_dict_data_s1Post = """{"satelliteParams": {"satelliteType": "Sentinel1","processingLevel": "LEVEL1","sensorMode": "IW","productType": "GRD"},"startDate":\"""" + str(floodDate) + """\","endDate": \"""" + str(s1_postFloodDate) + """\","geometry": \"""" + str(geometry) + """\","area_sampling": "True","tmp_path_same_folder_dwl":"True","artifact_name": "sentinel1_GRD_postflood"}"""
    string_dict_data_s2Pre =  """{"satelliteParams":{"satelliteType": "Sentinel2","processingLevel": "S2MSI2A","bandmath": ["NDWI"]},"startDate":\"""" + str(s2_preFloodDate) + """\","endDate": \"""" + str(floodDate) + """\","geometry": \"""" + str(geometry) + """\","cloudCover": "[0,20]","area_sampling": "True","artifact_name": "sentinel2_pre_flood","preprocess_data_only": "false"}"""
    string_dict_data_s2Post = """{"satelliteParams":{"satelliteType": "Sentinel2","processingLevel": "S2MSI2A","bandmath": ["NDWI"]},"startDate":\"""" + str(floodDate) + """\","endDate": \"""" + str(s2_postFloodDate) + """\","geometry": \"""" + str(geometry) + """\","cloudCover": "[0,20]","area_sampling": "True","artifact_name": "sentinel2_post_flood","preprocess_data_only": "false"}"""
           
    with pipeline_context() as pc:

        s1 = pc.step(name="downloadS1Pre",
                     function="download_images_s1",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s1Pre],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ])

        s2 = pc.step(name="downloadS1Post",
                     function="download_images_s1",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s1Post],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ]).after(s1)
        
        s3 = pc.step(name="downloadS2Pre",
                     function="download_images_s2",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s2Pre],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ]).after(s2)

        s4 = pc.step(name="downloadS2Post",
                     function="download_images_s2",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s2Post],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ]).after(s3)

        s5 = pc.step(name="elaborate",
                     function="elaborate",
                     action="job",
                     fs_group='8877',
                     resources={"cpu": {"requests": "3", "limits": "6"},"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "200Gi" }
                    }],
                     args=['/shared/launch.sh', 'sentinel1_GRD_preflood', 'sentinel1_GRD_postflood', 'sentinel2_pre_flood', 'sentinel2_post_flood', str(geometry), 'Slopes_TN', 'slope_map25832.tif', 'Lakes_TN', 'idrspacq.shp', 'Rivers_TN', 'cif_pta2022_v.shp', str(outputName), str(floodDate), 'EPSG:25832', "['VV','VH']", '700', '7', '15', '2']
                     ).after(s4)
     


Overwriting flood_pipeline.py


Create workflow using project repo source file

In [ ]:
workflow = proj.new_workflow(
name="pipeline_flood_gitversion",
kind="kfp",
code_src="git+https://<username>:<personal_access_token>@github.com/tn-aixpa/rs-deforestation",
handler="src.deforestation_pipeline:myhandler")

Build workflow

In [ ]:
wfbuild = workflow.run(action="build", wait=True)

Run workflow (Garda)

In [ ]:
workflow_run = workflow.run(action="pipeline", parameters={
    "geometry":"POLYGON ((10.644988646837982 45.85539621678084, 10.644988646837982 46.06780100571985, 10.991744628283294 46.06780100571985, 10.991744628283294 45.85539621678084, 10.644988646837982 45.85539621678084))",
    "outputName": "flood_mask_oct_2020",
    "floodDate":"2020-10-02",
    "s1_preFloodDate": "2020-09-25",
    "s1_postFloodDate": "2020-10-09",
    "s2_preFloodDate": "2020-09-12",
    "s2_postFloodDate": "2020-10-22"
    })

Run workflow (Vad di Non)

In [ ]:
workflow_run = workflow.run(action="pipeline", parameters={
    "geometry":"POLYGON ((10.640782493560744 46.42790226539165, 11.157037287805009 46.420279834833785, 11.064149949974212 46.03812698432117, 10.558816402631928 46.064237047456984, 10.640782493560744 46.42790226539165))",
    "outputName": "val_di_non_oct_2018",
    "floodDate":"2018-10-27",
    "s1_preFloodDate": "2018-10-20",
    "s1_postFloodDate": "2018-11-04",
    "s2_preFloodDate": "2018-10-07",
    "s2_postFloodDate": "2018-11-17"
    })